In [40]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torch.utils.data.sampler import SubsetRandomSampler

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [41]:
def data_loader(data_dir, batch_size, random_seed=42, valid_size=0.1, shuffle=True, test=False):
    normalize = transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010],
    )

    # Define transforms
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        normalize,
    ])

    if test:
        dataset = datasets.CIFAR10( root=data_dir, train=False, download=True, transform=transform)
        data_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
        return data_loader

    # load the dataset
    train_dataset = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=transform)
    valid_dataset = datasets.CIFAR10(root=data_dir, train=True,download=True, transform=transform)

    num_train = len(train_dataset)
    indices = list(range(num_train))
    split = int(np.floor(valid_size * num_train))

    if shuffle:
        np.random.seed(42)
        np.random.shuffle(indices)

    train_idx, valid_idx = indices[split:], indices[:split]
    train_sampler = SubsetRandomSampler(train_idx)
    valid_sampler = SubsetRandomSampler(valid_idx)

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, sampler=train_sampler)
    valid_loader = torch.utils.data.DataLoader(
        valid_dataset, batch_size=batch_size, sampler=valid_sampler)

    return (train_loader, valid_loader)


train_loader, valid_loader = data_loader(data_dir='./data', batch_size=64)
test_loader = data_loader(data_dir='./data', batch_size=64, test=True)

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1, downsampler = None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = stride, padding = 1),
                        nn.BatchNorm2d(out_channels),
                        nn.ReLU())
        self.conv2 = nn.Sequential(
                        nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1),
                        nn.BatchNorm2d(out_channels))
        # This is a 1x1 convolution to match the dimensions of the input and output
        # called Network in-Network (NiN) block
        self.channel_downsampler = downsampler
        self.relu = nn.ReLU()
        self.out_channels = out_channels

    def forward(self, x):
        residual = x
        if self.channel_downsampler:
            residual = self.channel_downsampler(x)
        out = self.conv1(x)
        out = self.conv2(out)
        out += residual
        out = self.relu(out)
        return out

In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes = 10):
        super(ResNet, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Sequential(
                        nn.Conv2d(3, 64, kernel_size = 7, stride = 2, padding = 3),
                        nn.BatchNorm2d(64),
                        nn.ReLU())
        self.maxpool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)
        
        self.layer0 = self._make_layer(block, 64, layers[0], stride = 1)
        self.layer1 = self._make_layer(block, 128, layers[1], stride = 2)
        self.layer2 = self._make_layer(block, 256, layers[2], stride = 2)
        self.layer3 = self._make_layer(block, 512, layers[3], stride = 2)

        self.avgpool = nn.AvgPool2d(7, stride=1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, blocks, stride=1):
        channel_downsampler = None
        if stride != 1 or self.inplanes != planes:
            channel_downsampler = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride),
                nn.BatchNorm2d(planes),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, channel_downsampler))
        self.inplanes = planes
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)

        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

In [44]:
num_classes = 10
num_epochs = 20
batch_size = 16
learning_rate = 0.1

model = ResNet(ResidualBlock, [3, 4, 6, 3]).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay = 0.001, momentum = 0.9)  

#Train the model
total_step = len(train_loader)

In [ ]:
import gc

from regex import F
total_step = len(train_loader)

model.train()
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        #Move tensors to the configured device
        images = images.to(device)
        labels = labels.to(device)

        #Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        #Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i % 100 == 0:
            # Print loss and accuracy
            with torch.no_grad():
                accuracy = (100 * (outputs.argmax(dim=1) == labels).float().mean())
                print(f"Batch = {i+1}/{total_step}, Epoch = {epoch+1}/{num_epochs}, Loss = {loss.item():.4f}, Accuracy = {accuracy:.2f}%")

        del images, labels, outputs
        torch.cuda.empty_cache()
        gc.collect()

model.val()
#Validation
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in valid_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        del images, labels, outputs

    print('Accuracy of the network on the {} validation images: {} %'.format(5000, 100 * correct / total))

Batch = 1/704, Epoch = 1/20, Loss = 2.5079, Accuracy = 7.81%
Batch = 101/704, Epoch = 1/20, Loss = 2.3872, Accuracy = 15.62%
Batch = 201/704, Epoch = 1/20, Loss = 2.0577, Accuracy = 17.19%
Batch = 301/704, Epoch = 1/20, Loss = 2.0402, Accuracy = 26.56%
Batch = 401/704, Epoch = 1/20, Loss = 1.8653, Accuracy = 20.31%
Batch = 501/704, Epoch = 1/20, Loss = 1.7886, Accuracy = 26.56%
Batch = 601/704, Epoch = 1/20, Loss = 1.7090, Accuracy = 34.38%
Batch = 701/704, Epoch = 1/20, Loss = 1.7090, Accuracy = 37.50%
Batch = 1/704, Epoch = 2/20, Loss = 1.7475, Accuracy = 28.12%
Batch = 101/704, Epoch = 2/20, Loss = 1.6551, Accuracy = 35.94%
Batch = 201/704, Epoch = 2/20, Loss = 1.6647, Accuracy = 34.38%
Batch = 301/704, Epoch = 2/20, Loss = 1.4646, Accuracy = 39.06%
Batch = 401/704, Epoch = 2/20, Loss = 1.4651, Accuracy = 43.75%
Batch = 501/704, Epoch = 2/20, Loss = 1.3323, Accuracy = 50.00%
Batch = 601/704, Epoch = 2/20, Loss = 1.9339, Accuracy = 28.12%
Batch = 701/704, Epoch = 2/20, Loss = 1.3170,

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x00000292C14C2600>>
Traceback (most recent call last):
  File "c:\Users\Nitin\miniconda3\Lib\site-packages\ipykernel\ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x00000292C14C2600>>
Traceback (most recent call last):
  File "c:\Users\Nitin\miniconda3\Lib\site-packages\ipykernel\ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


Batch = 301/704, Epoch = 6/20, Loss = 1.1689, Accuracy = 60.94%


In [ ]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        del images, labels, outputs

    print('Accuracy of the network on the {} test images: {} %'.format(10000, 100 * correct / total))